In [1]:
import requests
import pandas as pd
import time

all_subfields = []
page = 1

while True:
    url = f"https://api.openalex.org/subfields?per_page=200&page={page}"
    response = requests.get(url)
    
    if response.status_code != 200:
        break
        
    data = response.json().get('results', [])
    if not data:
        break
        
    all_subfields.extend(data)
    print(f"  - Fetched page {page} ({len(data)} subfields)")
    
    page += 1
    time.sleep(0.5) 

hierarchy_map = {}
for sf in all_subfields:
    sf_id = sf['id'].split('/')[-1] # e.g., "1100"
    
    field = sf.get('field', {})
    domain = sf.get('domain', {})
    
    hierarchy_map[sf_id] = {
        'Subfield': sf['display_name'],
        'ID': field.get('id', '').split('/')[-1],
        'Field': field.get('display_name', ''),
        'Domain': domain.get('display_name', '')
    }

print(f"Successfully built hierarchy for {len(hierarchy_map)} subfields.")
pd.DataFrame.from_dict(hierarchy_map, orient='index').head()



  - Fetched page 1 (200 subfields)
  - Fetched page 2 (52 subfields)
Successfully built hierarchy for 252 subfields.


,Subfield,ID,Field,Domain
3312,Sociology and Political Science,33,Social Sciences,Social Sciences
3106,Nuclear and High Energy Physics,31,Physics and Astronomy,Physical Sciences
1110,Plant Science,11,Agricultural and Biological Sciences,Life Sciences
2208,Electrical and Electronic Engineering,22,Engineering,Physical Sciences
1312,Molecular Biology,13,"Biochemistry, Genetics and Molecular Biology",Life Sciences


In [2]:
import requests
import pandas as pd
import collections
import time

# --- STEP 1: Build Hierarchy Map (Run Once) ---
print("Building taxonomy map...")
subfields_url = "https://api.openalex.org/subfields?per_page=200"
all_subfields = []
page = 1

while True:
    r = requests.get(f"{subfields_url}&page={page}")
    if r.status_code != 200: break
    data = r.json().get('results', [])
    if not data: break
    all_subfields.extend(data)
    page += 1
    time.sleep(0.1)

hierarchy_map = {}
for sf in all_subfields:
    sf_id = sf['id'].split('/')[-1]
    field = sf.get('field', {})
    domain = sf.get('domain', {}) 
    if not domain: domain = field.get('domain', {}) # Fallback

    hierarchy_map[sf_id] = {
        'Subfield': sf['display_name'],
        'Field': field.get('display_name', 'Unknown'),
        'Domain': domain.get('display_name', 'Unknown')
    }
print(f"Taxonomy built ({len(hierarchy_map)} subfields).")

# --- STEP 2: Loop Through Years ---
uva_id = "I51556381"
start_year = 2010
end_year = 2025

for year in range(start_year, end_year + 1):
    print(f"\n--- Processing Year: {year} ---")
    
    # Reset counters for this specific year
    subfield_counts = collections.Counter()
    total_works = 0
    cursor = "*"
    
    # Filter for ONE year
    filter_str = f"authorships.institutions.lineage:{uva_id},publication_year:{year}"
    
    while True:
        url = (
            f"https://api.openalex.org/works?"
            f"filter={filter_str}"
            f"&select=id,primary_topic"
            f"&per_page=200&cursor={cursor}"
        )
        
        try:
            r = requests.get(url)
            if r.status_code != 200: 
                print(f"API Error {r.status_code}")
                break
                
            data = r.json()
            results = data.get('results', [])
            
            if not results:
                break
                
            cursor = data['meta']['next_cursor']
            
            for work in results:
                topic = work.get('primary_topic')
                if topic and topic.get('subfield'):
                    sf_id = topic['subfield']['id'].split('/')[-1]
                    subfield_counts[sf_id] += 1
                else:
                    subfield_counts['Unclassified'] += 1
            
            total_works += len(results)
            # Optional: print progress for large years
            # if total_works % 5000 == 0: print(f"  Fetched {total_works}...")
            
        except Exception as e:
            print(f"Error: {e}")
            break
            
    print(f"Year {year} complete. Found {total_works} works.")

    # --- STEP 3: Create DataFrame for this Year ---
    rows = []
    for sf_id, count in subfield_counts.items():
        if sf_id == 'Unclassified': continue
            
        info = hierarchy_map.get(sf_id)
        if info:
            rows.append({
                'Domain': info['Domain'],
                'Group': info['Field'],      # Rename for your viz
                'Field': info['Subfield'],   # Rename for your viz
                'numpub': count
            })

    df = pd.DataFrame(rows)
    
    if not df.empty:
        df = df.sort_values(by=['Domain', 'Group', 'Field'])
        
        # --- OPTION A: Save to file (if running locally) ---
        filename = f"uva_data_{year}.csv"
        df.to_csv(filename, index=False)
        print(f"Saved {filename}")
    else:
        print(f"No data found for {year}")

Building taxonomy map...
Taxonomy built (252 subfields).

--- Processing Year: 2010 ---
Year 2010 complete. Found 4619 works.
Saved uva_data_2010.csv

--- Processing Year: 2011 ---
Year 2011 complete. Found 4865 works.
Saved uva_data_2011.csv

--- Processing Year: 2012 ---
Year 2012 complete. Found 4860 works.
Saved uva_data_2012.csv

--- Processing Year: 2013 ---
Year 2013 complete. Found 5819 works.
Saved uva_data_2013.csv

--- Processing Year: 2014 ---
Year 2014 complete. Found 5283 works.
Saved uva_data_2014.csv

--- Processing Year: 2015 ---
Year 2015 complete. Found 5455 works.
Saved uva_data_2015.csv

--- Processing Year: 2016 ---
Year 2016 complete. Found 5699 works.
Saved uva_data_2016.csv

--- Processing Year: 2017 ---
Year 2017 complete. Found 6958 works.
Saved uva_data_2017.csv

--- Processing Year: 2018 ---
Year 2018 complete. Found 6084 works.
Saved uva_data_2018.csv

--- Processing Year: 2019 ---
Year 2019 complete. Found 6541 works.
Saved uva_data_2019.csv

--- Processi